# Model 09: historically constrained founder scenarios

This notebook tests the debate's proposed ~11,000-year founder-pair scenario as a **time-dependent reproductive-connectivity problem**.

The historical dates constrain when paths can exist. The migration / parental-source probabilities remain sensitivity parameters unless explicitly supported by external data.

In [ ]:
import os, sys, subprocess
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/Evolution-Creation'):
        subprocess.run(['git','clone','-q','https://github.com/vafaei-ar/Evolution-Creation.git','/content/Evolution-Creation'], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','-e','/content/Evolution-Creation'], check=True)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from evolution_creation.historical_constraints import (
    deterministic_historical_genealogy,
    earliest_reachable_generations,
    make_debate_historical_scenario,
    simulate_historical_genealogy,
    simulate_historical_pedigree_genome,
    simulate_late_contact_sensitivity,
)


## Historical ordering

- Sahul was occupied tens of thousands of years before 11 ka.
- Human populations were established in the Americas before 11 ka.
- Recent syntheses place final Bassian land-bridge submergence around 12 ka, with a broader literature range around 10–13.5 ka depending on the criterion.
- Permanent British settlement in Tasmania began in 1803.

The isolation date is therefore a parameter rather than a fixed assumption.

In [ ]:
founder_age=widgets.IntSlider(value=11000,min=8000,max=14000,step=250,description='Founder age y')
years_per_generation=widgets.FloatSlider(value=25.0,min=20.0,max=35.0,step=0.5,description='Years/gen')
isolation_age=widgets.IntSlider(value=12000,min=9000,max=14000,step=250,description='Tas isolation')
late_contact_rate=widgets.FloatLogSlider(value=0.01,base=10,min=-4,max=-1,step=0.1,description='Late mix')
ordinary_rate=widgets.FloatLogSlider(value=0.003,base=10,min=-4,max=-1,step=0.1,description='Ordinary mix')
sahul_rate=widgets.FloatLogSlider(value=0.001,base=10,min=-5,max=-1,step=0.1,description='Sahul mix')
americas_rate=widgets.FloatLogSlider(value=0.0005,base=10,min=-5,max=-1,step=0.1,description='Americas mix')
endogamy=widgets.FloatSlider(value=0.0,min=0.0,max=1.0,step=0.05,description='Endogamy')
joint_children=widgets.IntSlider(value=2,min=0,max=8,step=1,description='Joint children')
seed=widgets.IntText(value=20260920,description='Seed')
display(founder_age,years_per_generation,isolation_age,late_contact_rate,ordinary_rate,sahul_rate,americas_rate,endogamy,joint_children,seed)

In [ ]:
def build_scenario(region_sizes=(50,60,70,55,30,60)):
    return make_debate_historical_scenario(
        founder_age_years=founder_age.value,
        years_per_generation=years_per_generation.value,
        region_sizes=region_sizes,
        tasmania_isolation_age_years=isolation_age.value,
        tasmania_late_contact_rate=late_contact_rate.value,
        ordinary_bridge_rate=ordinary_rate.value,
        sahul_bridge_rate=sahul_rate.value,
        americas_bridge_rate=americas_rate.value,
        endogamy_strength=endogamy.value,
        founder_pair_joint_children=joint_children.value,
    )

def run_deterministic(_=None):
    scenario=build_scenario()
    result=deterministic_historical_genealogy(scenario)
    fig,ax=plt.subplots(figsize=(11,6))
    for i,name in enumerate(scenario.region_names):
        ax.plot(result.generations,result.all_founders_fraction_by_region[:,i],label=name)
    ax.set(xlabel='Generations after founder insertion',ylabel='Fraction descended from both founders',ylim=(0,1.02))
    ax.legend(ncol=2); plt.show()

    reach=earliest_reachable_generations(scenario)
    for name,g in zip(scenario.region_names,reach):
        print(f'{name:16s}: ' + ('unreachable' if not np.isfinite(g) else f'first positive path at generation {int(g)}'))
    print('Isolation generation after founder insertion:',scenario.metadata['isolation_generation'])
    print('Late-contact window (generations):',scenario.metadata['late_contact_generations'])
    print('Late-contact window starts at generation:',scenario.metadata['late_contact_start_generation'])

det_button=widgets.Button(description='Run historical recursion',button_style='primary')
det_button.on_click(run_deterministic)
display(det_button)
run_deterministic()

## Finite-population stochastic run

This version samples individual parents. Founder lineages can disappear by chance. Simulation population sizes are effective computational sizes, not census estimates.

In [ ]:
def run_stochastic(_=None):
    scenario=build_scenario()
    result=simulate_historical_genealogy(scenario,seed=seed.value)
    tas=scenario.region_names.index('Tasmania')
    fig,ax=plt.subplots(figsize=(10,5))
    ax.plot(result.generations,result.global_all_founders_fraction,label='global: both founders')
    ax.plot(result.generations,result.all_founders_fraction_by_region[:,tas],label='Tasmania: both founders')
    ax.set(xlabel='Generation',ylabel='Fraction',ylim=(0,1.02))
    ax.legend(); plt.show()
    print('Global universal generation:',result.global_universal_generation)
    print('Founder extinction generations:',result.founder_extinction_generations)

stoch_button=widgets.Button(description='Run finite simulation')
stoch_button.on_click(run_stochastic)
display(stoch_button)

## Isolate the late-contact question

For this panel only, assume the external population is already 100% descended from both founders. This is deliberately favorable to the founder scenario and isolates how much ancestry can spread during a short reopening window.

In [ ]:
rates=np.array([0,0.0001,0.00025,0.0005,0.001,0.002,0.003,0.005,0.0075,0.01,0.015,0.02])
late=simulate_late_contact_sensitivity(rates,generations=9,population_size=100,replicates=5000,seed=seed.value)
fig,ax=plt.subplots(figsize=(10,5))
ax.plot(100*rates,late.mean_final_fraction,marker='o',label='mean final descendant fraction')
ax.plot(100*rates,late.fixation_probability,marker='o',label='complete-fixation probability')
ax.set(xlabel='External-parent probability per parental draw (%)',ylabel='Fraction / probability',ylim=(0,1.02))
ax.legend(); plt.show()


## Optional pedigree + chromosome diagnostic

This uses the same time-varying regional parent-source matrices but also transmits explicit founder-derived autosomal segments. Full 440-generation chromosome runs are heavier, so the default diagnostic uses smaller effective regional populations. Those sizes are computational and are **not** historical census estimates.

In [ ]:
genome_region_size=widgets.IntSlider(value=8,min=4,max=20,step=1,description='Size/region')
genome_generations=widgets.IntSlider(value=60,min=10,max=440,step=10,description='Genome gen')
threshold_cm=widgets.FloatSlider(value=6.0,min=0.0,max=20.0,step=0.5,description='Threshold cM')
display(genome_region_size,genome_generations,threshold_cm)

def run_genome(_=None):
    n=genome_region_size.value
    scenario=build_scenario(region_sizes=(n,n,n,n,n,n))
    g=min(genome_generations.value,scenario.generations)
    result=simulate_historical_pedigree_genome(
        scenario,
        max_generations=g,
        detectable_threshold_cm=threshold_cm.value,
        seed=seed.value,
    )
    fig,ax=plt.subplots(figsize=(10,5))
    for i,name in enumerate(scenario.region_names):
        ax.plot(result.generations,result.genetic_carrier_fraction_by_region[:,i],label=name)
    ax.set(xlabel='Generation',ylabel='Fraction carrying any founder-set autosomal DNA',ylim=(0,1.02))
    ax.legend(ncol=2); plt.show()

genome_button=widgets.Button(description='Run chromosome diagnostic')
genome_button.on_click(run_genome)
display(genome_button)

## Interpretation rule

A zero reproductive path is a hard impossibility for founder ancestry. A positive path only establishes possibility, not probability. Once a barrier reopens, the remaining generations, effective cross-group parentage, endogamy, population size, and founder-lineage survival determine how far genealogy spreads.

The bridge-rate sliders are not empirical estimates and should not be cited as historical migration rates.